# Exercise 05 — Efficient Frontier

MSc Finance · Investments · FHNW · Autumn 2026

Lecture 04 gave you a curve between two assets. Lecture 05 turned it into a machine:
feed in expected returns, volatilities and correlations, and out come the
minimum-variance frontier, the tangency portfolio, and a weight vector for every
investor. This exercise runs that machine on the five asset classes of slide 27, and
then does the one thing slide 25 warns about — it moves a single input by less than its
own standard error and watches what happens to the output.

**How to work with this notebook.** The task text is here and on the exercise sheet.
Each code cell is a stub: the `# TODO` lines are the steps, in order. The setup and data
cells below are complete — run them first and leave them alone. Tasks 1 and 2 are the
exercise; Task 3 is there for groups that get through them early, and we walk through
its result together whether or not you ran it.

**The optimizer is written for you.** The three solver functions between Task 1 and
Task 2 are complete, and the cell above them explains what each argument does. Sixty
minutes is not enough to learn `scipy.optimize` and mean-variance optimization at the
same time, so this week you learn the second by calling the first. Read that cell
carefully — the arguments are the economics.

Run **Runtime → Restart and run all** before you trust any number in here. Nothing in
this notebook draws random numbers, so every group's output is identical to the last
decimal.

**Data.** J.P. Morgan Asset Management, *2026 Long-Term Capital Market Assumptions*
(30th annual edition), Swiss franc assumption matrix, source data as of 30 September
2025 — see the
[LTCMA publication page](https://am.jpmorgan.com/ch/en/asset-management/institutional/insights/portfolio-insights/long-term-capital-market-assumptions/).
The file holds Swiss Cash and the five risky classes the lecture used, with returns and
volatilities in percent per annum and a 5×5 correlation matrix.

Two course assumptions are recorded in the `Note` column of the `Assumptions` sheet, and
both are worth understanding before you use the numbers. The published expected return
on Swiss Cash is 1.20 %, which sits *above* the 0.29 % expected return on Swiss
government bonds — a long-run average that is defensible on its own terms but leaves
cash dominating bonds outright, and a risk-free rate above the minimum-variance
portfolio's expected return. It is set to 0.10 %. The published volatility of cash,
0.41 %, is set to zero, so that cash is risk-free in the exact sense the model needs and
therefore carries no correlations.


In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from exercise_utils import FHNW, ASSET_CYCLE, setup_style, save_results
setup_style()


In [ ]:
BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_05/data/ltcma_2026_chf.xlsx"

# Two sheets: six lines of assumptions, and the correlation matrix of the five
# risky classes. Cash is risk-free here, so it carries no correlations.
cma = pd.read_excel(DATA_URL, sheet_name="Assumptions").set_index("Asset class")
corr = pd.read_excel(DATA_URL, sheet_name="Correlations", index_col=0)

cma = cma.rename(columns={"Arithmetic return 2026 (%)": "ER",
                          "Annualized volatility (%)": "SD"})
cma[["ER", "SD"]] /= 100.0                      # percent per annum -> decimals

RISKY = list(corr.index)                        # the five, in the file's order
mu = cma.loc[RISKY, "ER"].values                # expected returns
sd = cma.loc[RISKY, "SD"].values                # volatilities
rf = cma.loc["Swiss Cash", "ER"]                # the risk-free rate
N = len(RISKY)

print(f"{N} risky asset classes, risk-free rate {rf:.2%}")
for k, name in enumerate(RISKY):
    print(f"  {name:24s}  E(R) {mu[k]:6.2%}   sigma {sd[k]:6.2%}")


## Task 1 — Two portfolios, before any optimizer

Slide 27 put an optimized portfolio next to a conventional Swiss balanced mandate.
Before touching an optimizer, evaluate two portfolios you already know how to handle.

The file gives volatilities and correlations, not a covariance matrix. Build one:

$$\Sigma = D\,R\,D, \qquad D = \operatorname{diag}(\sigma_1, \dots, \sigma_5),
\qquad \text{so} \quad \Sigma_{ij} = \rho_{ij}\,\sigma_i\,\sigma_j .$$

From there, Lecture 04's matrix notation applies unchanged:

$$E(R_P) = \mathbf{w}'\boldsymbol{\mu}, \qquad
\sigma_P = \sqrt{\mathbf{w}'\Sigma\,\mathbf{w}}, \qquad
SR = \frac{E(R_P) - R_f}{\sigma_P}.$$

Write `portfolio_stats(w)` returning those three numbers — every later task calls it —
and evaluate two fully invested portfolios. The first is the Pictet-60 style mandate of
slide 27: 30 % Developed World Equity, 10 % EM Equity, 5 % Commodities, 20 % Swiss
Equity, 35 % Swiss Government Bonds. The second is the equally weighted $1/N$ portfolio.

Then answer the question the two numbers pose: **which of the two would a mean-variance
investor rather hold, and does the answer depend on how risk-averse that investor is?**

*Deliverable: a 2×3 table of E(R), σ and SR, and one sentence on the ranking.*


In [ ]:
# TODO: build Sigma from the volatilities and the correlation matrix

# TODO: portfolio_stats(w) -> Series with E(R), sigma and the Sharpe ratio
def portfolio_stats(w):
    ...

In [ ]:
W = {"Pictet-60 style": np.array([0.30, 0.10, 0.05, 0.20, 0.35]),
     "1/N":             np.repeat(1 / N, N)}

# TODO: one row of statistics per portfolio

print(comparison.to_string(float_format="{:.4f}".format))

### The optimizer you are given

The three functions below are complete. Read them before Task 2 — the arguments are the
economics, and you will be changing them rather than writing them.

`scipy.optimize.minimize(objective, w0, method="SLSQP", bounds=..., constraints=...)`
searches for the weight vector that makes `objective` as small as possible. Four pieces
matter:

- **`objective`** — a function of the weights returning one number. Minimizing
  $\mathbf{w}'\Sigma\mathbf{w}$ gives a minimum-variance portfolio; minimizing the
  *negative* Sharpe ratio maximizes it, since there is no `maximize`.
- **`w0`** — where the search starts. Equal weights, here.
- **`constraints`** — a list of dictionaries. `{"type": "eq", "fun": f}` demands
  $f(\mathbf{w}) = 0$. Being fully invested is $\sum_i w_i - 1 = 0$; hitting a target
  return is $\mathbf{w}'\boldsymbol{\mu} - \text{target} = 0$.
- **`bounds`** — a range per weight. This is where the short-sale ban lives: $(0, 1)$
  per asset forbids negative weights, and widening the range to $(-2, 3)$ permits them.
  Note that "long-only" is a *bounds* change, not an extra constraint.

`SLSQP` is the method that handles equality constraints and bounds together. It is a
local search, so it reports whether it converged; all three functions return `None`
rather than a wrong answer when it did not. A target return no portfolio can reach — say
12 % when the best single asset offers 8.65 % and shorting is banned — is exactly such a
case, and your code has to expect it.


In [ ]:
FULLY_INVESTED = {"type": "eq", "fun": lambda w: w.sum() - 1.0}
BOUNDS = {True: [(0.0, 1.0)] * N,        # long-only
          False: [(-2.0, 3.0)] * N}      # shorts and leverage permitted


def _solve(objective, long_only, extra=()):
    """Minimise `objective` over fully invested weights. None if SLSQP fails."""
    res = minimize(objective, np.repeat(1 / N, N), method="SLSQP",
                   bounds=BOUNDS[long_only],
                   constraints=[FULLY_INVESTED, *extra],
                   options={"ftol": 1e-14, "maxiter": 3000})
    return res.x if res.success else None


def min_variance_portfolio(long_only=True):
    """The MVP: lowest variance of all fully invested portfolios."""
    return _solve(lambda w: w @ Sigma @ w, long_only)


def min_variance_at(target, long_only=True):
    """Lowest variance among portfolios with expected return exactly `target`."""
    hit_target = {"type": "eq", "fun": lambda w: w @ mu - target}
    return _solve(lambda w: w @ Sigma @ w, long_only, (hit_target,))


def tangency_portfolio(mu_vec, long_only=True):
    """The maximum-Sharpe portfolio for the expected returns `mu_vec`."""
    return _solve(lambda w: -(w @ mu_vec - rf) / np.sqrt(w @ Sigma @ w), long_only)


## Task 2 — Trace the frontier, then find the tangency portfolio

Slide 8 described Markowitz as four steps. Steps 2 and 3 are now two function calls in a
loop.

Write `trace_frontier(long_only, top, n=120)`: for each of `n` target returns, minimize
variance subject to hitting that target, and collect the resulting volatility, Sharpe
ratio and gross exposure $\sum_i |w_i|$.

**Start the grid at the minimum-variance portfolio's expected return.** Below it, every
portfolio is dominated — there is another with the same volatility and a higher expected
return (slide 6) — and step 4 of the procedure is precisely to discard that branch.
Tracing from zero instead is the commonest way to end up presenting a "frontier" whose
lower half no investor would ever hold.

Trace twice: **long-only** up to the highest single-asset expected return, and
**unconstrained** up to 12 %. Then locate the tangency portfolio under each bounds
regime with `tangency_portfolio`, print the two side by side with their gross exposures,
and read the gross exposure along the unconstrained frontier at target returns of 4, 6,
8, 10 and 12 %.

Two questions to answer from the output. **What does the short-sale ban cost at the
tangency point?** And **what is the unconstrained frontier buying with its extra
exposure** as the target return rises?

*Deliverable: the figure, the two tangency portfolios side by side, and the
gross-exposure table.*


In [ ]:
def trace_frontier(long_only, top, n=120):
    # TODO: begin the grid at the MVP's expected return, not at zero

    # TODO: minimise variance at each target; skip any the solver cannot reach
    return pd.DataFrame(rows)

lo = trace_frontier(long_only=True,  top=mu.max())
un = trace_frontier(long_only=False, top=0.12)

In [ ]:
# TODO: the tangency portfolio under each bounds regime

# TODO: the two weight vectors as one table, and their statistics as another

print("weights, %")
print(tp_weights.round(1).to_string(), "\n")
print(tp_stats.to_string(float_format="{:.4f}".format))

# TODO: gross exposure along the unconstrained frontier at five target returns

In [ ]:
fig1, ax = plt.subplots(figsize=(7.4, 5.2))

# Long-only underneath, unconstrained dashed on top: where the two coincide the
# dashed line sits on the thick one, and the divergence above is unmistakable.
ax.plot(100 * lo["sigma"], 100 * lo["target"], color=FHNW["navy"], lw=4.0,
        solid_capstyle="round", label="Long-only frontier")
ax.plot(100 * un["sigma"], 100 * un["target"], color=FHNW["blue"], ls="--", lw=1.6,
        label="Unconstrained frontier")

s_tp = portfolio_stats(w_tp["long-only"])
cal_x = np.linspace(0, 100 * lo["sigma"].max() * 1.3, 50)
ax.plot(cal_x, 100 * rf + s_tp["SR"] * cal_x, color=FHNW["orange"], lw=1.4,
        label="CAL through the tangency portfolio")

ax.scatter(100 * sd, 100 * mu, s=34, color="#7A7A7A", zorder=3)
x_max = 100 * lo["sigma"].max() * 1.35
# Five fixed points, so the labels are placed by hand rather than by a rule that
# would put one of them on top of the frontier. "EM Equity" is the slide-27 name.
PLACE = {"Developed World Equity":  ("Developed World Equity", (9, -3), "left"),
         "Emerging Markets Equity": ("EM Equity", (0, -9), "center"),
         "Commodities":             ("Commodities", (0, -9), "center"),
         "Swiss Equity":            ("Swiss Equity", (0, -9), "center"),
         "Swiss Government Bonds":  ("Swiss Government Bonds", (9, -3), "left")}
for k, name in enumerate(RISKY):
    label, offset, align = PLACE[name]
    ax.annotate(label, (100 * sd[k], 100 * mu[k]), fontsize=8, color="#555555",
                ha=align, va="center" if align == "left" else "top",
                xytext=offset, textcoords="offset points")

w_mvp = min_variance_portfolio(long_only=True)
s_mvp = portfolio_stats(w_mvp)
ax.scatter([100 * s_mvp["sigma"]], [100 * s_mvp["E(R)"]], s=70, marker="s",
           color=FHNW["green"], zorder=4, label="MVP, long-only")
ax.scatter([100 * s_tp["sigma"]], [100 * s_tp["E(R)"]], s=130, marker="*",
           color=FHNW["red"], zorder=4, label="Tangency portfolio")
ax.scatter([0], [100 * rf], s=34, color=FHNW["orange"], zorder=4)

ax.set_xlabel("Volatility, % p.a.")
ax.set_ylabel("Expected return, % p.a.")
ax.set_xlim(0, x_max)
ax.set_ylim(0, 13)
ax.legend(loc="lower right", fontsize=9)


## Task 3 (optional) — Move one input by less than its standard error

*For groups that finish Tasks 1 and 2 with time to spare. Nothing later depends on it,
and we work through the result together in the walkthrough either way.*

Slide 25 put a number on how badly expected returns are known. Estimated from $T$ years
of data, the standard error of a mean return is

$$SE = \frac{\sigma}{\sqrt{T}},$$

so twenty-five years of Developed World Equity at $\sigma = 16.14\ \%$ pins its
expected return down to about **3.2 percentage points** either way. Now perturb the
inputs by a fraction of that.

Take each of the five asset classes in turn, move its expected return by
$\pm 0.5$ percentage points — roughly one sixth of that standard error — and re-solve
the **long-only** tangency portfolio. Ten re-solves against the base case, one table.

Report the resulting weight table, and for each bump the largest single weight change
against the base case. Put the five standard errors beside it, so the comparison is
between how much the input was allowed to move and how much the output actually did.

Then answer: **is the sensitivity the same for every asset, and if not, what explains
the differences?**

*Deliverable: the weight table, the largest shift per bump, and the standard errors.*


In [ ]:
BUMP = 0.005

# TODO: re-solve the long-only tangency with one expected return moved by +-BUMP

bumps = pd.DataFrame(weights, index=RISKY).T

# TODO: the largest single weight change against the base case, per bump

print((100 * bumps).round(1).to_string())

In [ ]:
# TODO: the standard error of a 25-year mean estimate, in percentage points

print("largest weight shift, percentage points")
print((100 * shift).round(1).sort_values(ascending=False).to_string())
print("\nstandard error of a 25-year mean estimate, percentage points")
print((100 * se_25y).round(2).to_string())

fig2, ax = plt.subplots(figsize=(7.4, 4.4))
order = shift.sort_values()
colours = [FHNW["navy"] if s.endswith("+0.5pp") else FHNW["blue"]
           for s in order.index]
ax.barh(range(len(order)), 100 * order.values, color=colours)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order.index, fontsize=8)
ax.set_xlabel("Largest weight change against the base case, percentage points")
handles = [plt.Rectangle((0, 0), 1, 1, color=FHNW["navy"]),
           plt.Rectangle((0, 0), 1, 1, color=FHNW["blue"])]
ax.legend(handles, ["E(R) raised by 0.5pp", "E(R) lowered by 0.5pp"],
          loc="lower right", fontsize=9)

## Export

Bundle the figures and the tables, in case you want them for the transfer questions or
your own notes.


In [ ]:
# TODO: export the frontier figure with the Task 1 and Task 2 tables

if "fig2" in globals():                       # Task 3 was done
    figures["error_maximization"] = fig2
    tables["bumps"] = bumps

save_results(figures=figures, tables=tables, name="ex05")

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Every number in this notebook was forward-looking. The expected returns came from a
published set of assumptions, the tangency portfolio was optimal *given* them, and
nothing here was ever tested against a realised return. Task 3 showed how little those
assumptions can be trusted; it did not show what that costs. Lecture 06 supplies the
missing piece — the index model, which replaces an estimated covariance matrix with a
structured one — and Exercise 06 runs the horse race the theory has been asking for:
$1/N$ against sample mean-variance against index-model mean-variance, on twenty Swiss
stocks, out of sample.
